In [ ]:
import io
import math
import timeit
import itertools
import functools
import contextlib
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter, deque
from typing import List, Tuple, Dict, Any, Callable

# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q140)
# ==========================================
# Question 140:
# Using itertools exclusively, write a function that: (a) generates all combinations of k items from a list 
# without repetition, (b) produces a rolling window of size n over an iterable, (c) groups consecutive identical 
# elements, and (d) computes the Cartesian product of three lists.
#
# Sample Input:  sample_list = [1, 1, 2, 3, 3, 3, 4], k = 2, n = 3, lists = ([1, 2], ['a', 'b'], [True])
# Sample Output: {'combinations': [(1, 1), (1, 2), ...], 'rolling_window': [(1, 1, 2), (1, 2, 3), ...], 'consecutive_groups': [(1, [1, 1]), (2, [2]), ...], 'cartesian_product': [(1, 'a', True), ...]}

def itertools_toolkit(data: list, k: int, n: int, list1: list, list2: list, list3: list) -> dict:
    # (a) Combinations of k items without repetition
    combs = list(itertools.combinations(data, k))
    
    # (b) Rolling window of size n over an iterable
    iters = itertools.tee(data, n)
    for i, it in enumerate(iters):
        for _ in range(i):
            next(it, None)
    rolling = list(zip(*iters))
    
    # (c) Groups consecutive identical elements
    groups = [(k_val, list(g)) for k_val, g in itertools.groupby(data)]
    
    # (d) Cartesian product of three lists
    cart_prod = list(itertools.product(list1, list2, list3))
    
    return {
        'combinations': combs,
        'rolling_window': rolling,
        'consecutive_groups': groups,
        'cartesian_product': cart_prod
    }

if __name__ == '__main__':
    # Test Question 140
    data_in = [1, 1, 2, 3, 3, 3, 4]
    res = itertools_toolkit(data_in, 2, 3, [1, 2], ['a', 'b'], [True])
    print("Q140 Output Combinations Count:", len(res['combinations']))
    print("Q140 Output Rolling Window:", res['rolling_window'])
    print("Q140 Output Groups:", res['consecutive_groups'])
    print("Q140 Output Cartesian Product:", res['cartesian_product'])





In [ ]:

# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q141)
# ==========================================
# Question 141:
# Use collections.Counter to write a function that takes two documents (long strings) and produces a 
# cosine-similarity score based on word frequencies. Normalise by document length.
#
# Sample Input:  doc1 = "data science and machine learning", doc2 = "machine learning and deep learning"
# Sample Output: 0.73

def cosine_similarity(doc1: str, doc2: str) -> float:
    words1 = doc1.lower().split()
    words2 = doc2.lower().split()
    
    vec1 = Counter(words1)
    vec2 = Counter(words2)
    
    intersection = set(vec1.keys()) & set(vec2.keys())
    dot_product = sum(vec1[w] * vec2[w] for w in intersection)
    
    mag1 = math.sqrt(sum(v ** 2 for v in vec1.values()))
    mag2 = math.sqrt(sum(v ** 2 for v in vec2.values()))
    
    if not mag1 or not mag2:
        return 0.0
    return round(dot_product / (mag1 * mag2), 2)

if __name__ == '__main__':
    # Test Question 141
    d1 = "data science and machine learning"
    d2 = "machine learning and deep learning"
    print("Q141 Output:", cosine_similarity(d1, d2))



In [ ]:
# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q142)
# ==========================================
# Question 142:
# Use collections.deque to implement a fixed-size sliding-window statistics tracker: as each new value 
# is added, it automatically drops the oldest, and exposes mean, min, max, and standard deviation of the 
# current window. Demonstrate with a stream of 100 values and window=10.
#
# Sample Input:  stream = range(1, 101), window_size = 10
# Sample Output: {'mean': 95.5, 'min': 91, 'max': 100, 'stdev': 2.87}

class SlidingWindowStats:
    def __init__(self, window_size: int = 10):
        self.window = deque(maxlen=window_size)

    def add(self, val: float):
        self.window.append(val)

    def stats(self) -> dict:
        if not self.window:
            return {'mean': 0.0, 'min': 0.0, 'max': 0.0, 'stdev': 0.0}
        n = len(self.window)
        mean_val = sum(self.window) / n
        min_val = min(self.window)
        max_val = max(self.window)
        variance = sum((x - mean_val) ** 2 for x in self.window) / n
        stdev_val = math.sqrt(variance)
        return {
            'mean': round(mean_val, 2),
            'min': min_val,
            'max': max_val,
            'stdev': round(stdev_val, 2)
        }

if __name__ == '__main__':
    # Test Question 142
    tracker = SlidingWindowStats(window_size=10)
    for x in range(1, 101):
        tracker.add(x)
    print("Q142 Output:", tracker.stats())


In [ ]:


# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q143)
# ==========================================
# Question 143:
# Use functools.lru_cache to memoize a recursive solution for the number of ways to make change for 
# a given amount using a given list of coin denominations. Compare execution time with and without the 
# cache using timeit.
#
# Sample Input:  amount = 50, coins = (1, 2, 5, 10, 25)
# Sample Output: (ways: 292, cached_time < uncached_time)

def count_change_uncached(amount: int, coins: tuple) -> int:
    def helper(amt, idx):
        if amt == 0: return 1
        if amt < 0 or idx >= len(coins): return 0
        return helper(amt - coins[idx], idx) + helper(amt, idx + 1)
    return helper(amount, 0)

def count_change_cached(amount: int, coins: tuple) -> int:
    @functools.lru_cache(maxsize=None)
    def helper(amt, idx):
        if amt == 0: return 1
        if amt < 0 or idx >= len(coins): return 0
        return helper(amt - coins[idx], idx) + helper(amt, idx + 1)
    return helper(amount, 0)

if __name__ == '__main__':
    # Test Question 143
    test_amt = 50
    test_coins = (1, 2, 5, 10, 25)
    
    t_uncached = timeit.timeit(lambda: count_change_uncached(test_amt, test_coins), number=20)
    t_cached = timeit.timeit(lambda: count_change_cached(test_amt, test_coins), number=20)
    
    ways = count_change_cached(test_amt, test_coins)
    print("Q143 Output:", {
        'ways': ways,
        'uncached_time_sec': round(t_uncached, 5),
        'cached_time_sec': round(t_cached, 5)
    })

In [ ]:
# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q144)
# ==========================================
# Question 144:
# Rewrite the Employee, Manager, and Product classes from earlier sections as dataclasses using 
# @dataclass. Use field(default_factory=...) for mutable defaults, add __post_init__ validation, and mark 
# appropriate fields as frozen. Show that frozen fields cannot be modified.
#
# Sample Input:  Product(id=1, name="Laptop", price=999.9), Manager(id=101, name="Alice", team=[...])
# Sample Output: {'product_name': 'Laptop', 'manager_team_size': 1, 'frozen_immutable': True}

@dataclass(frozen=True)
class ProductDC:
    id: int
    name: str
    price: float

    def __post_init__(self):
        if self.price < 0:
            raise ValueError("Price cannot be negative")

@dataclass
class EmployeeDC:
    id: int
    name: str
    skills: List[str] = field(default_factory=list)

@dataclass
class ManagerDC(EmployeeDC):
    team: List[EmployeeDC] = field(default_factory=list)

if __name__ == '__main__':
    # Test Question 144
    prod = ProductDC(1, "Laptop", 999.9)
    emp = EmployeeDC(2, "Bob", ["Python", "SQL"])
    mgr = ManagerDC(101, "Alice", ["Management"], team=[emp])
    
    is_frozen = False
    try:
        prod.price = 799.9  # Should raise FrozenInstanceError
    except Exception:
        is_frozen = True

    print("Q144 Output:", {
        'product_name': prod.name,
        'manager_team_size': len(mgr.team),
        'frozen_immutable': is_frozen
    })

In [ ]:

# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q143)
# ==========================================
# Question 143:
# Use functools.lru_cache to memoize a recursive solution for the number of ways to make change for 
# a given amount using a given list of coin denominations. Compare execution time with and without the 
# cache using timeit.
#
# Sample Input:  amount = 50, coins = (1, 2, 5, 10, 25)
# Sample Output: (ways: 292, cached_time < uncached_time)

def count_change_uncached(amount: int, coins: tuple) -> int:
    def helper(amt, idx):
        if amt == 0: return 1
        if amt < 0 or idx >= len(coins): return 0
        return helper(amt - coins[idx], idx) + helper(amt, idx + 1)
    return helper(amount, 0)

def count_change_cached(amount: int, coins: tuple) -> int:
    @functools.lru_cache(maxsize=None)
    def helper(amt, idx):
        if amt == 0: return 1
        if amt < 0 or idx >= len(coins): return 0
        return helper(amt - coins[idx], idx) + helper(amt, idx + 1)
    return helper(amount, 0)

if __name__ == '__main__':
    # Test Question 143
    test_amt = 50
    test_coins = (1, 2, 5, 10, 25)
    
    t_uncached = timeit.timeit(lambda: count_change_uncached(test_amt, test_coins), number=20)
    t_cached = timeit.timeit(lambda: count_change_cached(test_amt, test_coins), number=20)
    
    ways = count_change_cached(test_amt, test_coins)
    print("Q143 Output:", {
        'ways': ways,
        'uncached_time_sec': round(t_uncached, 5),
        'cached_time_sec': round(t_cached, 5)
    })




In [ ]:

# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q145)
# ==========================================
# Question 145:
# Use pathlib.Path to write a function that recursively scans a directory, groups files by extension, 
# computes total size per extension (in MB), and returns a sorted dict. Handle permission errors gracefully 
# with a warning.
#
# Sample Input:  directory_path = "."
# Sample Output: {'.py': 0.05, '.txt': 0.01}

def scan_directory_extensions(directory_path: str = ".") -> dict:
    base_path = Path(directory_path)
    extension_sizes = {}
    
    try:
        for p in base_path.rglob("*"):
            try:
                if p.is_file():
                    ext = p.suffix.lower() if p.suffix else "no_extension"
                    size_mb = p.stat().st_size / (1024 * 1024)
                    extension_sizes[ext] = extension_sizes.get(ext, 0.0) + size_mb
            except (PermissionError, OSError):
                continue
    except (PermissionError, OSError):
        pass

    # Sort extensions descending by size
    return {k: round(v, 4) for k, v in sorted(extension_sizes.items(), key=lambda item: item[1], reverse=True)}

if __name__ == '__main__':
    # Test Question 145
    print("Q145 Output:", scan_directory_extensions("."))


In [ ]:

# ==========================================
# 13. ITERTOOLS, COLLECTIONS & STDLIB (Selected: Q146)
# ==========================================
# Question 146:
# Use contextlib.suppress, contextlib.ExitStack, and contextlib.redirect_stdout to write: (a) a block that 
# silently ignores FileNotFoundError, (b) a function that opens an arbitrary number of files safely, and (c) a 
# function that captures the stdout of another function and returns it as a string. Show all three working 
# together.
#
# Sample Input:  run_context_suite(["temp1.txt", "temp2.txt"])
# Sample Output: {'captured_stdout': 'Running task...\nDone.\n', 'clean_execution': True}

def open_multiple_files_safely(file_paths: List[str]):
    with contextlib.ExitStack() as stack:
        files = [stack.enter_context(open(fp, 'w', encoding='utf-8')) for fp in file_paths]
        for idx, f in enumerate(files):
            f.write(f"Payload from file {idx}\n")
    return len(file_paths)

def capture_stdout(func: Callable, *args, **kwargs) -> str:
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        func(*args, **kwargs)
    return buffer.getvalue()

def run_context_suite(file_paths: List[str]) -> dict:
    # (a) Silently suppress missing file error
    with contextlib.suppress(FileNotFoundError):
        Path("non_existent_file_xyz.txt").unlink()

    # (b) & (c) Capture stdout of file-opening task
    def task():
        print("Running task...")
        count = open_multiple_files_safely(file_paths)
        print(f"Successfully wrote {count} files.")
        print("Done.")

    output = capture_stdout(task)
    
    # Cleanup created temp files
    for fp in file_paths:
        with contextlib.suppress(FileNotFoundError):
            Path(fp).unlink()

    return {'captured_stdout': output, 'clean_execution': True}

if __name__ == '__main__':
    # Test Question 146
    test_files = ["test_ctx_1.txt", "test_ctx_2.txt"]
    print("Q146 Output:", run_context_suite(test_files))